# Model Architecture before and after Quantization

In [1]:
# !pip install peft

In [5]:
# Imports

import os
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datetime import datetime
from peft import LoraConfig, PeftModel

In [4]:
# Constants

BASE_MODEL = 'meta-llama/Llama-3.2-3B'

# Below fine-tuned model is provided by course instructor, Ed Donner, Imported here for Architecture and LoRA Parameter Overview:
FINETUNED_MODEL = f"ed-donner/price-2025-11-30_15.10.55-lite"

In [6]:
# Login to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Trying out Different Quantization on Base Model:

### 1. Base Model without Quantization:

In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    device_map= 'auto')

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9:,.2f}GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory FootPrint of 'meta-llama/Llama-3.2-3B' is 6.43GB


In [13]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

### 2. Base Model with 8bit Quantization:

In [14]:
quant_config = BitsAndBytesConfig(
    load_in_8bit= True
)

base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9:,.2f}GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory FootPrint of 'meta-llama/Llama-3.2-3B' is 3.61GB


In [15]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear8bitLt(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (nor

### Base Model with 4bit Quantization:

In [17]:
quant_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    bnb_4bit_quant_type= 'nf4'
)

base_model= AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9:,.2f}GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory FootPrint of 'meta-llama/Llama-3.2-3B' is 2.20GB


In [18]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

## Fine-Tuned Model:

In [19]:
fine_tuned_model = PeftModel.from_pretrained(model= base_model, model_id= FINETUNED_MODEL)

print(f"Memory FootPrint of '{FINETUNED_MODEL}' is {fine_tuned_model.get_memory_footprint() / 1e9:,.2f}GB")

adapter_config.json: 0.00B [00:00, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--ed-donner--price-2025-11-30_15.10.55-lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


adapter_model.safetensors:   0%|          | 0.00/73.4M [00:00<?, ?B/s]

Memory FootPrint of 'ed-donner/price-2025-11-30_15.10.55-lite' is 2.27GB


In [20]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Understanding LoRA Weights and Dimensions:

### 1. LoRA Setup to use for lite version of dataset:

In [21]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha * lora_A * lora_B
# Let's count the number of weights using their dimensions:

r = 32

# See the matrix dimensions above

# Attention layers
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# Each layer comes to
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# There are 28 layers
params = lora_layer * 28

# So the total size in MB is
size = (params * 4) / 1_000_000

print(f"Total number of params: {params:,} and size {size:,.1f}MB")

Total number of params: 18,350,080 and size 73.4MB
